<a target="_blank" href="https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/week_14/week14_sqlite_supabase_postgres_admin_demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Week 14 Demo: SQLite Ideas That Transfer To Supabase/Postgres

This notebook is a small database administration demo.

It uses SQLite because SQLite runs anywhere without cloud credentials. The ideas transfer to Supabase/Postgres:

- create tables
- insert seed data
- write useful queries
- create an index
- inspect a query plan
- export a backup-style SQL script

SQLite is not Supabase. Supabase uses Postgres. Some syntax is different, but the database thinking is the same.

## 1. Create A Tiny Final Project Database

Imagine a student is building a simple support ticket tracker.

The database needs to track:

- users
- tickets
- ticket status
- ticket priority

This is a good Supabase/Postgres project because the data is structured and relational.

In [ ]:
import sqlite3
from pathlib import Path

db_path = Path("week14_support_tickets_demo.sqlite")

if db_path.exists():
    db_path.unlink()

conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("PRAGMA foreign_keys = ON;")

cur.executescript("""
CREATE TABLE users (
    user_id INTEGER PRIMARY KEY,
    full_name TEXT NOT NULL,
    role TEXT NOT NULL CHECK (role IN ('student', 'staff'))
);

CREATE TABLE tickets (
    ticket_id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL,
    title TEXT NOT NULL,
    status TEXT NOT NULL CHECK (status IN ('open', 'in_progress', 'closed')),
    priority TEXT NOT NULL CHECK (priority IN ('low', 'medium', 'high')),
    created_at TEXT NOT NULL,
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
""")

conn.commit()
print(f"Created demo database: {db_path}")

### Supabase/Postgres Parallel

In Supabase, the same design would be Postgres tables. The SQL would be very similar, but Postgres would commonly use types such as `serial`, `bigserial`, `identity`, `text`, `date`, and `timestamptz`.

The important final project evidence is not the exact SQLite syntax. The important evidence is that the student can explain tables, columns, primary keys, relationships, and constraints.

## 2. Insert Seed Data

Seed data proves the model works.

A final project should have enough rows to test useful queries.

In [ ]:
users = [
    (1, "Avery Rivera", "student"),
    (2, "Jordan Lee", "student"),
    (3, "Morgan Patel", "staff"),
    (4, "Sam Chen", "staff"),
]

tickets = [
    (1, 1, "Cannot access lab VM", "open", "high", "2026-05-01"),
    (2, 1, "Password reset request", "closed", "medium", "2026-05-02"),
    (3, 2, "Supabase connection issue", "in_progress", "high", "2026-05-03"),
    (4, 2, "Need project feedback", "open", "medium", "2026-05-04"),
    (5, 1, "Dataset upload failed", "open", "high", "2026-05-06"),
    (6, 2, "Question about indexes", "closed", "low", "2026-05-07"),
]

cur.executemany("INSERT INTO users VALUES (?, ?, ?);", users)
cur.executemany("INSERT INTO tickets VALUES (?, ?, ?, ?, ?, ?);", tickets)
conn.commit()

print("Inserted seed data")

## 3. Write Useful Queries

A final project should answer realistic questions.

Bad query evidence: `SELECT * FROM tickets;`

Better query evidence: `Which high-priority tickets are still open or in progress?`

In [ ]:
query = """
SELECT
    tickets.ticket_id,
    tickets.title,
    tickets.status,
    tickets.priority,
    tickets.created_at,
    users.full_name
FROM tickets
JOIN users ON tickets.user_id = users.user_id
WHERE tickets.priority = 'high'
  AND tickets.status != 'closed'
ORDER BY tickets.created_at DESC;
"""

rows = cur.execute(query).fetchall()
for row in rows:
    print(dict(row))

## 4. Create A Reporting Query

Reporting queries summarize data.

This maps to the Week 13 aggregation idea: group data and count results.

In [ ]:
report_query = """
SELECT
    status,
    COUNT(*) AS ticket_count
FROM tickets
GROUP BY status
ORDER BY ticket_count DESC;
"""

for row in cur.execute(report_query):
    print(dict(row))

### MongoDB Parallel

The SQL reporting query above is similar to a MongoDB aggregation pipeline like this:

```javascript
db.tickets.aggregate([
  { $group: { _id: "$status", ticket_count: { $sum: 1 } } },
  { $sort: { ticket_count: -1 } }
])
```

Different syntax. Same idea: group records and count them.

## 5. Add One Index Decision

Indexes should support real queries.

Our useful query filters by `priority` and `status`, then sorts by `created_at`.

A reasonable beginner index is on those columns.

In [ ]:
index_sql = """
CREATE INDEX idx_tickets_priority_status_created
ON tickets (priority, status, created_at);
"""

cur.execute(index_sql)
conn.commit()

print("Created index: idx_tickets_priority_status_created")

### Supabase/Postgres Parallel

In Supabase SQL Editor, the Postgres version would look similar:

```sql
CREATE INDEX idx_tickets_priority_status_created
ON tickets (priority, status, created_at);
```

Final project explanation:

`I chose this index because users often search for high-priority tickets that are still open or in progress, ordered by date. The index may help that read pattern. The cost is extra storage and slightly slower writes because the index must be maintained.`

## 6. Inspect The Query Plan

SQLite has `EXPLAIN QUERY PLAN`.

Postgres has `EXPLAIN` and `EXPLAIN ANALYZE`.

The exact output is different, but the purpose is similar: ask the database how it plans to run the query.

In [ ]:
plan = cur.execute("EXPLAIN QUERY PLAN " + query).fetchall()

for step in plan:
    print(tuple(step))

## 7. Backup-Style Export

A backup plan should explain what gets saved and how restore would be checked.

SQLite can export SQL with `iterdump()`.

Supabase/Postgres projects might use SQL files, CSV exports, dashboard backups, CLI tools, or platform backups depending on the project.

In [ ]:
backup_path = Path("week14_support_tickets_backup.sql")

with backup_path.open("w", encoding="utf-8") as f:
    for line in conn.iterdump():
        f.write(line + "\n")

print(f"Wrote backup-style SQL export: {backup_path}")
print("First 15 lines:")
print("\n".join(backup_path.read_text(encoding="utf-8").splitlines()[:15]))

## 8. Restore Verification Checklist

A restore is not finished until it is checked.

A beginner checklist could be:

- database opens
- tables exist
- row counts look right
- important query still works
- index still exists
- access-control concern is still understood

This checklist is exactly the kind of admin evidence students can include in the final project.

In [ ]:
checks = {
    "users_count": cur.execute("SELECT COUNT(*) FROM users;").fetchone()[0],
    "tickets_count": cur.execute("SELECT COUNT(*) FROM tickets;").fetchone()[0],
    "open_tickets": cur.execute("SELECT COUNT(*) FROM tickets WHERE status = 'open';").fetchone()[0],
}

indexes = cur.execute("PRAGMA index_list('tickets');").fetchall()

print("Verification checks:")
for name, value in checks.items():
    print(f"- {name}: {value}")

print("\nIndexes on tickets:")
for index in indexes:
    print(tuple(index))

## Final Project Writing Template

Students can adapt this paragraph:

> My project uses Supabase/Postgres because the data is structured and relational. The main tables are users and tickets. The database includes seed data so I can test realistic support-ticket queries. One important query finds high-priority tickets that are not closed and orders them by date. I would index priority, status, and created_at because that query filters and sorts on those fields. The cost is that inserts and updates require the index to be maintained. For access control, students should be able to create tickets, but only staff should close them. For restore, I would recover the database from an export or backup and verify table counts, important queries, and indexes.

That is the kind of admin evidence Week 14 is asking for.